In [0]:
# Importing

from pyspark.sql import functions as F

In [0]:
# Importing csv file

df = spark.read.csv("/Volumes/samples/databricks/datasets/online_retail/data-001/data.csv", header=True, inferSchema=True)

In [0]:
# Checking the data

df.show(5)
df.printSchema()

+---------+---------+--------------------+--------+------------+---------+----------+--------------+
|InvoiceNo|StockCode|         Description|Quantity| InvoiceDate|UnitPrice|CustomerID|       Country|
+---------+---------+--------------------+--------+------------+---------+----------+--------------+
|   536365|   85123A|WHITE HANGING HEA...|       6|12/1/10 8:26|     2.55|     17850|United Kingdom|
|   536365|    71053| WHITE METAL LANTERN|       6|12/1/10 8:26|     3.39|     17850|United Kingdom|
|   536365|   84406B|CREAM CUPID HEART...|       8|12/1/10 8:26|     2.75|     17850|United Kingdom|
|   536365|   84029G|KNITTED UNION FLA...|       6|12/1/10 8:26|     3.39|     17850|United Kingdom|
|   536365|   84029E|RED WOOLLY HOTTIE...|       6|12/1/10 8:26|     3.39|     17850|United Kingdom|
+---------+---------+--------------------+--------+------------+---------+----------+--------------+
only showing top 5 rows
root
 |-- InvoiceNo: string (nullable = true)
 |-- StockCode: strin

In [0]:
# Saving bronze table

spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.retail_project")

df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("retail_project.bronze")

In [0]:
# Checking null values

df_null = df.select([F.count(F.when(F.col(i).isNull(),i)).alias(i) for i in df.columns])
df_null.show()

+---------+---------+-----------+--------+-----------+---------+----------+-------+
|InvoiceNo|StockCode|Description|Quantity|InvoiceDate|UnitPrice|CustomerID|Country|
+---------+---------+-----------+--------+-----------+---------+----------+-------+
|        0|        0|        166|       0|          0|        0|     25281|      0|
+---------+---------+-----------+--------+-----------+---------+----------+-------+



In [0]:
# Checking where the Unit Price is 0

df.select("UnitPrice").where(df.UnitPrice <= 0).count()

337

In [0]:
# Cleaning the data

df_clean = df.withColumn("InvoiceDate", F.to_timestamp("InvoiceDate", 'M/d/yy H:m')).withColumn("Cancelled", F.when(F.col("Quantity") < 0, True).otherwise(False)).dropDuplicates()

In [0]:
# Checking cleaned data

df_clean.show(5)
df_clean.printSchema()

+---------+---------+--------------------+--------+-------------------+---------+----------+--------------+---------+
|InvoiceNo|StockCode|         Description|Quantity|        InvoiceDate|UnitPrice|CustomerID|       Country|Cancelled|
+---------+---------+--------------------+--------+-------------------+---------+----------+--------------+---------+
|   536365|    21730|GLASS STAR FROSTE...|       6|2010-12-01 08:26:00|     4.25|     17850|United Kingdom|    false|
|   536370|    22659|LUNCH BOX I LOVE ...|      24|2010-12-01 08:45:00|     1.95|     12583|        France|    false|
|   536373|   82494L|WOODEN FRAME ANTI...|       6|2010-12-01 09:02:00|     2.55|     17850|United Kingdom|    false|
|   536375|    22752|SET 7 BABUSHKA NE...|       2|2010-12-01 09:32:00|     7.65|     17850|United Kingdom|    false|
|   536377|    22633|HAND WARMER UNION...|       6|2010-12-01 09:34:00|     1.85|     17850|United Kingdom|    false|
+---------+---------+--------------------+--------+-----

In [0]:
# Checking count of data

df_clean.count()

64861

In [0]:
df_clean.distinct().count()

64861

In [0]:
# Saving silver table

df_clean.write \
  .format("delta") \
  .mode("overwrite") \
  .saveAsTable("retail_project.silver")

In [0]:
# Reading the table for querying

df_query = spark.read.table("retail_project.silver")

In [0]:
# Querying the top products by revenue

df_top_products = df_query.where(F.col("Cancelled") == False).groupBy("StockCode", "Description").agg(F.round(F.sum(F.col("Quantity") * F.col("UnitPrice")), 2).alias("Revenue")).filter(F.col("StockCode").rlike(r"\d")).orderBy("Revenue", ascending=False).limit(5)

df_top_products.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("retail_project.gold_top_products")

df_top_products.show()

+---------+--------------------+--------+
|StockCode|         Description| Revenue|
+---------+--------------------+--------+
|    23166|MEDIUM CERAMIC TO...|154367.2|
|    22423|REGENCY CAKESTAND...|73032.32|
|   85123A|WHITE HANGING HEA...|47495.76|
|    79321|       CHILLI LIGHTS|25650.92|
|   84029E|RED WOOLLY HOTTIE...|18861.74|
+---------+--------------------+--------+



In [0]:
# Querying the revenue per day

df_revenue_per_day = df_query.where(F.col("Cancelled") == False).groupBy(F.to_date("InvoiceDate").alias("Day")).agg(F.round(F.sum(F.col("Quantity") * F.col("UnitPrice")), 2).alias("Revenue")).orderBy("Day")

df_revenue_per_day.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("retail_project.gold_revenue_per_day")

df_revenue_per_day.show()

+----------+---------+
|       Day|  Revenue|
+----------+---------+
|2010-12-01|117553.58|
|2010-12-02| 95258.84|
|2010-12-03| 93797.26|
|2010-12-05| 62729.26|
|2010-12-06| 109248.3|
|2010-12-07| 199107.7|
|2010-12-08| 90470.72|
|2010-12-09|107096.38|
|2010-12-10|118042.04|
|2010-12-12|  34251.3|
|2010-12-13| 75884.74|
|2010-12-14| 90335.12|
|2010-12-15| 60768.74|
|2010-12-16| 98228.28|
|2010-12-17| 90676.26|
|2010-12-19| 14820.94|
|2010-12-20| 53532.46|
|2010-12-21| 94559.78|
|2010-12-22| 12399.94|
|2010-12-23| 24143.82|
+----------+---------+
only showing top 20 rows


In [0]:
# Querying the cancellation revenue over the total revenue

df_cancellation_rate = spark.sql("SELECT ROUND(ABS((SELECT SUM(Quantity * UnitPrice) FROM retail_project.silver WHERE Cancelled = TRUE)) / (SELECT SUM(Quantity * UnitPrice) FROM retail_project.silver WHERE Cancelled = FALSE) * 100, 2) AS Percentage")

df_cancellation_rate.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("retail_project.gold_cancellation_rate")

df_cancellation_rate.show()

+----------+
|Percentage|
+----------+
|     15.03|
+----------+

